# 10.1 长上下文扩展 (Context Extension)

扩展模型能处理的最大序列长度，使模型能处理长文档、多轮对话等场景。

本节涵盖：
- 位置插值（PI）
- NTK-Aware缩放
- YaRN

## 1. 位置插值（Position Interpolation）

**目的**：将预训练的短上下文模型扩展到更长上下文

**基本原理**：将长序列的位置索引线性映射到预训练时的位置范围内，使模型能处理超出训练长度的序列。

**核心公式**：
- 原始位置：[0, L_train-1]
- 扩展位置：[0, L_target-1]
- 映射：pos_new = pos_old × (L_train / L_target)
- 缩放因子：s = L_target / L_train

**代表**：LLaMA-2 4K→16K扩展

In [ ]:
import torch
import math

torch.manual_seed(42)

L_train = 2048
L_target = 16384
scale_factor = L_target / L_train

def original_rope(pos, d_head=64, base=10000):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
    freqs = pos * inv_freq
    return torch.cat([freqs.cos(), freqs.sin()])

def pi_rope(pos, d_head=64, base=10000, scale=8.0):
    scaled_pos = pos / scale
    return original_rope(scaled_pos, d_head, base)

positions = torch.arange(0, 16384, 1024).float()
print('=== Position Interpolation ===')
print(f'Training length: {L_train}, Target length: {L_target}')
print(f'Scale factor: {scale_factor:.1f}x')
print(f'\nPosition mapping (target -> interpolated):')
for p in positions[:8]:
    interp_p = p / scale_factor
    print(f'  pos={int(p):>5d} -> interp_pos={interp_p:.1f}')

print(f'\nKey: PI compresses positions into the trained range.')
print(f'No fine-tuning needed for small extensions (2-4x).')
print(f'Fine-tuning recommended for larger extensions (8x+).')

## 2. NTK-Aware缩放与YaRN

**NTK-Aware缩放**：调整RoPE的base频率而非线性缩放位置，高频分量保持不变，低频分量被拉伸，效果优于简单位置插值。

**YaRN**：结合NTK缩放和注意力温度调整，对不同频率分量使用不同策略，进一步改善长上下文性能。

**三种方法对比**：
| 方法 | 策略 | 微调需求 | 效果 |
|------|------|----------|------|
| PI | 线性缩放位置 | 少量 | 基础 |
| NTK-Aware | 调整base频率 | 无/少量 | 更好 |
| YaRN | NTK+注意力温度 | 少量 | 最好 |

In [ ]:
import torch
import math

torch.manual_seed(42)

L_train = 2048
scale = 8.0
d_head = 64
base_original = 10000

base_ntk = base_original * (scale ** (d_head / (d_head - 2)))

def ntk_rope(pos, d_head=64, base=10000):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
    freqs = pos * inv_freq
    return freqs

pos = torch.tensor(16000.0)
freqs_original = ntk_rope(pos, d_head, base_original)
freqs_pi = ntk_rope(pos / scale, d_head, base_original)
freqs_ntk = ntk_rope(pos, d_head, base_ntk)

print('=== NTK-Aware Scaling vs PI ===')
print(f'Position: {pos.item():.0f}, Scale: {scale}x')
print(f'NTK base: {base_ntk:.0f} (original: {base_original})')
print(f'\nFrequency comparison (first 8 dimensions):')
print(f'  Original (OOB):  {freqs_original[:8].tolist()}')
print(f'  PI:              {freqs_pi[:8].tolist()}')
print(f'  NTK-Aware:       {freqs_ntk[:8].tolist()}')
print(f'\nKey differences:')
print(f'  PI: all frequencies scaled equally (loses high-freq info)')
print(f'  NTK: high frequencies preserved, low frequencies stretched')
print(f'  YaRN: NTK + attention temperature adjustment for best results')
print(f'\nYaRN adds a temperature factor sqrt(s) to attention scores')
print(f'for tokens beyond the original training length.')